In [ ]:
#SET BASE DIRECTORY
#This notebook expects the GSE135779 data folders (adult_individual_h5ad, GSE135779_RAW, Results, etc.) to sit one level above this Notebooks folder. Update BASE_DIR below if your data lives elsewhere.

import os
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path(BASE_DIR) / "Notebooks"))
from publication_utils import configure_publication_notebook

FIGURE_DIR = configure_publication_notebook(BASE_DIR, "02C_GSE135779_ADULT_HEALTHY_LIANA")


**Donor-level LIANA inference.** LIANA consensus rank aggregation is run independently for each donor using `rank_aggregate.by_sample`, donor ID as `sample_key`, cell type as `groupby`, the consensus human resource, `expr_prop=0.1`, and a minimum of 10 cells per donor-cell-type combination. This preserves donor-level replication for downstream statistical tests.


In [ ]:
import scanpy as sc

adata_path = f"{BASE_DIR}/adult_individual_h5ad/adata_adult_healthy_liana.h5ad"

adata = sc.read_h5ad(adata_path)

**Donor-cell-type eligibility.** LIANA is run independently by donor. Each donor-cell-type combination must contain at least 10 cells; smaller combinations are excluded by the prespecified LIANA `min_cells` rule and reported in `sample_cell_type_counts.csv`.


In [ ]:
from analysis_config import MIN_CELLS_PER_SAMPLE_CELL_TYPE

sample_cell_counts = (
    adata.obs.groupby(["sample", "cell_type"], observed=True)
    .size().rename("n_cells").reset_index()
)
sample_cell_counts.to_csv(
    FIGURE_DIR / "sample_cell_type_counts.csv", index=False
)
below_floor = sample_cell_counts[
    sample_cell_counts["n_cells"] < MIN_CELLS_PER_SAMPLE_CELL_TYPE
]
print(f"Sample-cell-type combinations below the {MIN_CELLS_PER_SAMPLE_CELL_TYPE}-cell floor: {len(below_floor)}")


In [ ]:
import liana as li
from analysis_config import MIN_CELLS_PER_SAMPLE_CELL_TYPE

res = li.mt.rank_aggregate.by_sample(
    adata,
    groupby="cell_type",
    sample_key="sample",
    min_cells=MIN_CELLS_PER_SAMPLE_CELL_TYPE,
    resource_name="consensus",
    expr_prop=0.1,
    use_raw=False,
    verbose=False
)

print("\n--- After LIANA ---")
print("adata_adult.uns keys:", adata.uns.keys())

In [ ]:
res = adata.uns["liana_res"].copy()

print("Shape:", res.shape)
print("Columns:", res.columns.tolist())
print("Unique sources:", res["source"].unique())

res.head(10)

In [ ]:
from pathlib import Path
res = adata.uns["liana_res"].copy()
res["condition"] = "aH"

save_dir = f"{BASE_DIR}/Results/aH_liana"

Path(save_dir).mkdir(parents=True, exist_ok=True)

csv_path = f"{save_dir}/aH_liana_res.csv"
# LIANA results are retained as a table; a duplicate AnnData copy is unnecessary.
# h5ad_path = f"{save_dir}/aH_with_liana.h5ad"

res.to_csv(csv_path, index=False)

print("Saved:")
print(csv_path)


**Descriptive heatmap.** Mean `lrscore` per source-target cell-type pair across donor-level interaction rows. This visualization is descriptive; between-group inference is performed in notebooks 04A-04F and 05.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# path to your file
file_path = f"{BASE_DIR}/Results/aH_liana/aH_liana_res.csv"

# load data
res = pd.read_csv(file_path)

# build heatmap matrix
heatmap = res.groupby(["source", "target"])["lrscore"].mean().unstack().fillna(0)

# plot heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(heatmap, cmap="Reds")

plt.title("aH Cell–Cell Communication")
plt.xlabel("Target")
plt.ylabel("Source")

# save
save_path = f"{BASE_DIR}/Results/aH_liana/aH_heatmap.png"
plt.tight_layout()
plt.savefig(save_path, dpi=600)

plt.show()

print("Saved to:", save_path)